In [1]:
import pandas as pd
import numpy as np
import json
bio= '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/Enrollees.txt'
file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/AllClinical00.txt'
second_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/kxr_sq_bu00.txt'
third_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/Outcomes99.txt'
fourth_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/kxr_qjsw_duryea00.txt'
fifth_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/kmri_sq_moaks_bicl00.txt'
bio_df = pd.read_csv(bio, sep='|', encoding='latin1')
df = pd.read_csv(file_path, sep='|', encoding='latin1')
df_2 = pd.read_csv(second_file_path, sep='|', encoding='latin1')
df_3 = pd.read_csv(third_file_path, sep='|', encoding='latin1')
df_4 = pd.read_csv(fourth_file_path, sep='|', encoding='latin1')
df_5 = pd.read_csv(fifth_file_path, sep='|', encoding='latin1')


/Users/filippofocaccia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
clinical= df.copy()
clinical2= df_2.copy()
clinical3= df_3.copy()
clinical4= bio_df.copy()
clinical5= df_5.copy()

In [3]:

# Load MOAKS variable list and filter clinical5
with open('../MOAKS.json', 'r') as f:
    moaks_vars = json.load(f)

def flatten_vars(obj):
    """Recursively flatten all variable name strings from a nested dict/list."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        result = []
        for v in obj.values():
            result.extend(flatten_vars(v))
        return result
    return []

all_moaks_vars = flatten_vars(moaks_vars["VARIABLES"])

# Keep only variables that actually exist in the file
existing_moaks_vars = [v for v in all_moaks_vars if v in clinical5.columns]
missing_moaks_vars  = [v for v in all_moaks_vars if v not in clinical5.columns]
if missing_moaks_vars:
    print(f"Variables in MOAKS.json not found in file ({len(missing_moaks_vars)}): {missing_moaks_vars}")

clinical5 = clinical5[['ID', 'SIDE'] + existing_moaks_vars]
print(f"clinical5 shape: {clinical5.shape}")
clinical5.head()


Variables in MOAKS.json not found in file (2): ['V00OSPI', 'V00OSPM']
clinical5 shape: (5118, 115)


,ID,SIDE,V00MCMPM,V00MCMPL,V00MCMFMA,V00MCMFLA,V00MCMFMC,V00MCMFLC,V00MCMFMP,V00MCMFLP,...,V00MGCPCL,V00MGCTIB,V00MGCSM,V00MGSST,V00MGCOTH,V00MANSBUR,V00MIPBUR,V00MPPBUR,V00MPOPCYS,V00MITBSIG
0,9000622,1: Right,"2.1: 10-75% area, 1-10% full thickness",0: Normal,0: Normal,0: Normal,0: Normal,0: Normal,0: Normal,"1.1: 1-10% area, 1-10% full thickness",...,0: No,0: No,0: No,0: No,0: No,0: No,0: No,0: No,0: No,0: No
1,9000798,2: Left,"2: 10-75% area, no full thickness",0: Normal,"2: 10-75% area, no full thickness","2: 10-75% area, no full thickness","3.2: >75% area, 10-75% full thickness","1.1: 1-10% area, 1-10% full thickness","2.2: 10-75% area, 10-75% full thickness",0: Normal,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook
2,9001400,1: Right,"2.2: 10-75% area, 10-75% full thickness","2: 10-75% area, no full thickness","2: 10-75% area, no full thickness","2: 10-75% area, no full thickness",0: Normal,0: Normal,0: Normal,0: Normal,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook
3,9001695,1: Right,"2: 10-75% area, no full thickness",0: Normal,"1: 1-10% area, no full thickness",0: Normal,"2: 10-75% area, no full thickness","2: 10-75% area, no full thickness","1: 1-10% area, no full thickness","2: 10-75% area, no full thickness",...,0: No,0: No,0: No,0: No,0: No,0: No,0: No,0: No,1: Yes,.: Missing Form/Incomplete Workbook
4,9001897,1: Right,"2.1: 10-75% area, 1-10% full thickness","1: 1-10% area, no full thickness","1: 1-10% area, no full thickness",0: Normal,"2: 10-75% area, no full thickness",0: Normal,0: Normal,0: Normal,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook


In [4]:
#i want for each variable in clinical2 to create two new columns in clinical, one for the right side and one for the left side
for col in clinical5.columns:
    if col != 'ID' and col != 'SIDE':
        clinical5[f'{col}_R'] = clinical5.apply(lambda row: row[col] if row['SIDE'] == '1: Right' else None, axis=1)
        clinical5[f'{col}_L'] = clinical5.apply(lambda row: row[col] if row['SIDE'] == '2: Left' else None, axis=1)
        # Drop the original columns
        clinical5.drop(columns=[col], inplace=True)
clinical5.drop(columns=['SIDE'], inplace=True)
clinical5= clinical5.groupby("ID").first().reset_index()


/var/folders/9d/sqxp5kj56832p1mf59mk01gc0000gn/T/ipykernel_79522/522025288.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clinical5[f'{col}_R'] = clinical5.apply(lambda row: row[col] if row['SIDE'] == '1: Right' else None, axis=1)
/var/folders/9d/sqxp5kj56832p1mf59mk01gc0000gn/T/ipykernel_79522/522025288.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clinical5[f'{col}_L'] = clinical5.apply(lambda row: row[col] if row['SIDE'] == '2: Left' else None, axis=1)
/var/folders/9d/sqxp5kj56832p1mf59mk01gc0000gn/T/ipykernel_7952

In [5]:

def moaks_str_to_numeric(val):
    """Extract the numeric prefix from MOAKS string codes.
    
    Examples:
      '0: Normal'                        -> 0.0
      '1.1: 1-10% area, 1-10% full...'  -> 1.1
      '3: >75% area, no full thickness'  -> 3.0
      '0: No' / '1: Yes'                 -> 0.0 / 1.0
      '.: Missing Form/Incomplete...'    -> NaN
    """
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    s = str(val).strip()
    prefix = s.split(":")[0].strip()
    if prefix == ".":
        return np.nan
    try:
        return float(prefix)
    except ValueError:
        return np.nan

moaks_cols = [c for c in clinical5.columns if c != 'ID']
for col in moaks_cols:
    clinical5[col] = clinical5[col].apply(moaks_str_to_numeric)

print(clinical5.dtypes.value_counts())
clinical5.head()


float64    226
int64        1
Name: count, dtype: int64


,ID,V00MCMPM_R,V00MCMPM_L,V00MCMPL_R,V00MCMPL_L,V00MCMFMA_R,V00MCMFMA_L,V00MCMFLA_R,V00MCMFLA_L,V00MCMFMC_R,...,V00MANSBUR_R,V00MANSBUR_L,V00MIPBUR_R,V00MIPBUR_L,V00MPPBUR_R,V00MPPBUR_L,V00MPOPCYS_R,V00MPOPCYS_L,V00MITBSIG_R,V00MITBSIG_L
0,9000622,2.1,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN
1,9000798,NaN,2.0,NaN,0.0,NaN,2.0,NaN,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9001400,2.2,NaN,2.0,NaN,2.0,NaN,2.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9001695,2.0,NaN,0.0,NaN,1.0,NaN,0.0,NaN,2.0,...,0.0,NaN,0.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN
4,9001897,2.1,2.2,1.0,2.0,1.0,3.2,0.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Split clinical5 into right and left datasets based on column suffixes
r_moaks_cols = [c for c in clinical5.columns if c.endswith('_R')]
l_moaks_cols = [c for c in clinical5.columns if c.endswith('_L')]

clinical5_R = clinical5[['ID'] + r_moaks_cols].copy()
clinical5_L = clinical5[['ID'] + l_moaks_cols].copy()

print(f"clinical5_R shape: {clinical5_R.shape}")
print(f"clinical5_L shape: {clinical5_L.shape}")
# Drop rows where all MOAKS columns are NaN (no data for that side)
clinical5_R = clinical5_R.dropna(subset=r_moaks_cols, how='all').reset_index(drop=True)
clinical5_L = clinical5_L.dropna(subset=l_moaks_cols, how='all').reset_index(drop=True)

print(f"clinical5_R shape after dropping all-NaN rows: {clinical5_R.shape}")
print(f"clinical5_L shape after dropping all-NaN rows: {clinical5_L.shape}")

clinical5_R shape: (3229, 114)
clinical5_L shape: (3229, 114)
clinical5_R shape after dropping all-NaN rows: (2514, 114)
clinical5_L shape after dropping all-NaN rows: (1447, 114)


In [17]:
clinical5_R.isnull().sum().sort_values(ascending=False)
#drop columns which have missing data over 250
cols_to_drop_R = [col for col in clinical5_R.columns if clinical5_R[col].isnull().sum() > 250]
clinical5_R.drop(columns=cols_to_drop_R, inplace=True)
print(f"Dropped {len(cols_to_drop_R)} columns from clinical5_R. New shape: {clinical5_R.shape}")

Dropped 15 columns from clinical5_R. New shape: (2514, 40)


In [15]:
clinical5_L.isnull().sum().sort_values(ascending=False)
#drop columns which have missing data over 300
cols_to_drop_L = [col for col in clinical5_L.columns if clinical5_L[col].isnull().sum() > 300]
clinical5_L.drop(columns=cols_to_drop_L, inplace=True)
print(f"Dropped {len(cols_to_drop_L)} columns from clinical5_L. New shape: {clinical5_L.shape}")

Dropped 74 columns from clinical5_L. New shape: (1447, 40)


In [18]:
clinical5_R.isnull().sum().sort_values(ascending=False)


V00MBMSFLP_R    6
V00MBMSSS_R     6
V00MBMSFLA_R    5
V00MBMSFLC_R    5
V00MMTLP_R      5
V00MMTLA_R      4
V00MMTMP_R      4
V00MBMSPL_R     4
V00MCMPM_R      4
V00MBMSTLP_R    4
V00MBMSTLC_R    4
V00MBMSTLA_R    4
V00MBMSFMA_R    4
V00MBMSTMA_R    4
V00MBMSPM_R     4
V00MMTMA_R      3
V00MBMSTMP_R    3
V00MBMSTMC_R    3
V00MMXMM_R      3
V00MMXLL_R      3
V00MBMSFMP_R    3
V00MCMFLP_R     3
V00MCMPL_R      3
V00MCMFMA_R     3
V00MBMSFMC_R    3
V00MCMFLC_R     3
V00MCMTLA_R     2
V00MCMTMA_R     2
V00MCMTMC_R     2
V00MCMFMP_R     2
V00MCMTLC_R     2
V00MMTMB_R      2
V00MCMFMC_R     2
V00MCMFLA_R     2
V00MMTLB_R      2
V00MCMTMP_R     2
V00MMRTM_R      2
V00MMRTL_R      2
V00MCMTLP_R     2
ID              0
dtype: int64

In [19]:
clinical5_L.isnull().sum().sort_values(ascending=False)

V00MMXLL_L      3
V00MCMPL_L      3
V00MMXMM_L      2
V00MCMTLC_L     1
V00MCMPM_L      1
V00MBMSPM_L     1
V00MBMSPL_L     1
V00MMTMA_L      1
V00MCMTLP_L     1
V00MMTMP_L      1
V00MMTLA_L      1
V00MCMTLA_L     1
V00MMTLP_L      1
V00MCMFMC_L     1
V00MCMFLA_L     1
V00MBMSTLC_L    0
V00MMTLB_L      0
V00MMTMB_L      0
V00MMRTM_L      0
V00MBMSSS_L     0
V00MMRTL_L      0
V00MBMSTLP_L    0
V00MBMSTMP_L    0
ID              0
V00MBMSTMC_L    0
V00MBMSTLA_L    0
V00MBMSTMA_L    0
V00MBMSFMP_L    0
V00MBMSFLC_L    0
V00MBMSFMC_L    0
V00MBMSFLA_L    0
V00MBMSFMA_L    0
V00MCMTMP_L     0
V00MCMTMC_L     0
V00MCMTMA_L     0
V00MCMFLP_L     0
V00MCMFMP_L     0
V00MCMFLC_L     0
V00MCMFMA_L     0
V00MBMSFLP_L    0
dtype: int64

In [20]:
# Fill missing values with the most frequent value (mode) for each column
for col in clinical5_R.columns:
    if col != 'ID' and clinical5_R[col].isnull().any():
        clinical5_R[col] = clinical5_R[col].fillna(clinical5_R[col].mode()[0])

for col in clinical5_L.columns:
    if col != 'ID' and clinical5_L[col].isnull().any():
        clinical5_L[col] = clinical5_L[col].fillna(clinical5_L[col].mode()[0])

print("Remaining NaNs in clinical5_R:", clinical5_R.isnull().sum().sum())
print("Remaining NaNs in clinical5_L:", clinical5_L.isnull().sum().sum())

Remaining NaNs in clinical5_R: 0
Remaining NaNs in clinical5_L: 0


In [25]:
print('right columns:',list(clinical5_R.columns))
print('left columns:',list(clinical5_L.columns))

right columns: ['ID', 'V00MCMPM_R', 'V00MCMPL_R', 'V00MCMFMA_R', 'V00MCMFLA_R', 'V00MCMFMC_R', 'V00MCMFLC_R', 'V00MCMFMP_R', 'V00MCMFLP_R', 'V00MCMTMA_R', 'V00MCMTLA_R', 'V00MCMTMC_R', 'V00MCMTLC_R', 'V00MCMTMP_R', 'V00MCMTLP_R', 'V00MBMSFMA_R', 'V00MBMSFLA_R', 'V00MBMSFMC_R', 'V00MBMSFLC_R', 'V00MBMSFMP_R', 'V00MBMSFLP_R', 'V00MBMSTMA_R', 'V00MBMSTLA_R', 'V00MBMSTMC_R', 'V00MBMSTLC_R', 'V00MBMSTMP_R', 'V00MBMSTLP_R', 'V00MBMSPM_R', 'V00MBMSPL_R', 'V00MBMSSS_R', 'V00MMTMA_R', 'V00MMTMB_R', 'V00MMTMP_R', 'V00MMTLA_R', 'V00MMTLB_R', 'V00MMTLP_R', 'V00MMRTM_R', 'V00MMRTL_R', 'V00MMXMM_R', 'V00MMXLL_R']
left columns: ['ID', 'V00MCMPM_L', 'V00MCMPL_L', 'V00MCMFMA_L', 'V00MCMFLA_L', 'V00MCMFMC_L', 'V00MCMFLC_L', 'V00MCMFMP_L', 'V00MCMFLP_L', 'V00MCMTMA_L', 'V00MCMTLA_L', 'V00MCMTMC_L', 'V00MCMTLC_L', 'V00MCMTMP_L', 'V00MCMTLP_L', 'V00MBMSFMA_L', 'V00MBMSFLA_L', 'V00MBMSFMC_L', 'V00MBMSFLC_L', 'V00MBMSFMP_L', 'V00MBMSFLP_L', 'V00MBMSTMA_L', 'V00MBMSTLA_L', 'V00MBMSTMC_L', 'V00MBMSTLC_L', 'V00

In [26]:
clinical5_L.to_csv('../csv/MOAK_L.csv', index=False)
clinical5_R.to_csv('../csv/MOAK_R.csv', index=False)

In [13]:
# Filter the dataframe to keep only the columns specified in VARIABLE_DESCRIPTIONS
# Load the JSON file
with open('../variables_t.json', 'r') as file:
	variables = json.load(file)
columns_to_keep_first= ['ID']
columns_to_keep_second= ['ID','SIDE']
columns_to_keep_third= ['id']
columns_to_keep_fourth= ['ID']

# Extract the columns to keep from the allclinicall00 file for the symptoms (WOMAC and KOOS)
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_R"]["SYMPTOMS"])
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_L"]["SYMPTOMS"][:2])
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_L"]["BIO"][:2])
#extract the columns to keep from the kxr_sq_bu00 file for the structure and kl grade
columns_to_keep_second.extend(variables["VARIABLES"]["ALL_R"]["STRUCTURE"])
columns_to_keep_second.extend(variables["VARIABLES"]["ALL_R"]["KL_GRADE"])

#extract the columns to keep from the outcomes99 file for the surgery
columns_to_keep_third.extend(variables["VARIABLES"]["ALL_R"]["SURGERY"])
columns_to_keep_third.extend(variables["VARIABLES"]["ALL_L"]["SURGERY"])

columns_to_keep_fourth.extend(variables["VARIABLES"]["ALL_R"]["BIO"][2:])
clinical = clinical[columns_to_keep_first]
clinical2 = clinical2[columns_to_keep_second]
clinical3 = clinical3[columns_to_keep_third]
clinical4 = clinical4[columns_to_keep_fourth]

In [14]:
clinical4['P02SEX'] = clinical4['P02SEX'].apply(lambda x: 1 if x == '1: Male' else 2)
#1 for male patients and 2 for female patients

In [18]:
#abdominal circumference
clinical['V00ABCIRC'] = clinical['V00ABCIRC'].fillna(clinical['V00ABCIRC'].mean())

In [ ]:
#i want for each variable in clinical2 to create two new columns in clinical, one for the right side and one for the left side
for col in clinical2.columns:
    if col != 'ID' and col != 'SIDE':
        clinical2[f'{col}_R'] = clinical2.apply(lambda row: row[col] if row['SIDE'] == '1: Right' else None, axis=1)
        clinical2[f'{col}_L'] = clinical2.apply(lambda row: row[col] if row['SIDE'] == '2: Left' else None, axis=1)
        # Drop the original columns
        clinical2.drop(columns=[col], inplace=True)
clinical2.drop(columns=['SIDE'], inplace=True)
clinical2= clinical2.groupby("ID").first().reset_index()


,ID,V00XRJSL_R,V00XRJSL_L,V00XRJSM_R,V00XRJSM_L,V00XRSCFM_R,V00XRSCFM_L,V00XRSCFL_R,V00XRSCFL_L,V00XRSCTM_R,...,V00XROSFM_R,V00XROSFM_L,V00XROSFL_R,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L
0,9000099,0.0,2.0,0.0,0.0,0: 0,0: 0,0: 0,2: 2,0: 0,...,0: 0,0: 0,2: 2,2: 2,1: 1,0: 0,1: 1,1: 1,2: 2,3: 3
1,9000296,0.0,0.0,1.0,2.0,0: 0,0: 0,0: 0,0: 0,0: 0,...,0: 0,0: 0,0: 0,0: 0,1: 1,1: 1,0: 0,0: 0,2: 2,3: 3
2,9000622,0.0,0.0,0.0,0.0,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1: 1,1: 1
3,9000798,0.0,0.0,1.0,3.0,0: 0,2: 2,0: 0,0: 0,0: 0,...,0: 0,3: 3,0: 0,2: 2,0: 0,2: 2,0: 0,2: 2,1: 1,4: 4
4,9001104,0.0,0.0,2.0,1.0,2: 2,0: 0,0: 0,0: 0,1: 1,...,2: 2,0: 0,0: 0,0: 0,2: 2,0: 0,0: 0,0: 0,3: 3,1: 1


In [21]:
clinical3.rename(columns={'id': 'ID'}, inplace=True)
#now we can merge the three dataframes on the ID column
merged_df = clinical.merge(clinical2, on='ID', how='inner').merge(clinical3, on='ID', how='inner').merge(clinical4, on='ID', how='inner')
merged_df.head()

,ID,V00WOMTSR,V00KOOSKPR,V00KOOSYMR,V00KOOSQOL,V00WOMTSL,V00KOOSKPL,V00AGE,V00ABCIRC,V00XRJSL_R,...,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L,V99ERKVSAF,V99ELKVSAF,P02SEX
0,9000099,14.0,77.8,67.9,25.0,0.0,100.0,59,96.8,0.0,...,2: 2,1: 1,0: 0,1: 1,1: 1,2: 2,3: 3,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
1,9000296,0.0,100.0,100.0,100.0,0.0,100.0,69,104.3,0.0,...,0: 0,1: 1,1: 1,0: 0,0: 0,2: 2,3: 3,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
2,9000622,20.9,75.0,82.1,50.0,0.0,100.0,71,98.9,0.0,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1: 1,1: 1,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,2
3,9000798,0.0,100.0,100.0,43.8,30.0,59.4,56,109.0,0.0,...,2: 2,0: 0,2: 2,0: 0,2: 2,1: 1,4: 4,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
4,9001104,33.0,69.4,60.7,37.5,14.0,100.0,72,111.1,0.0,...,0: 0,2: 2,0: 0,0: 0,0: 0,3: 3,1: 1,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,2


In [22]:
#i want 0 for Missing Form/Incomplete Workbook and 1 for everything else
merged_df['V99ELKVSAF'] = merged_df['V99ELKVSAF'].apply(lambda x: 0 if x == '.: Missing Form/Incomplete Workbook' else 1)
merged_df['V99ERKVSAF']= merged_df['V99ERKVSAF'].apply(lambda x: 0 if x == '.: Missing Form/Incomplete Workbook' else 1)
#let's  see how many missing values we have in each column for the kl
merged_df['V00XRKL_R'] = merged_df['V00XRKL_R'].apply(lambda x: np.nan if x == '.: Missing Form/Incomplete Workbook' else x)
merged_df['V00XRKL_L'] = merged_df['V00XRKL_L'].apply(lambda x: np.nan if x == '.: Missing Form/Incomplete Workbook' else x)

# Replace missing values with the mean for the specified columns in the symptomes
mean_columns = ['V00WOMTSR', 'V00WOMTSL', 'V00KOOSKPL', 'V00KOOSKPR', 'V00KOOSQOL']
for col in mean_columns:
    merged_df[col] = merged_df[col].fillna(round(clinical[col].mean()))

#KL is very much specular between left and right, so we can fill the missing values accordingly
merged_df["V00XRKL_R"] = merged_df["V00XRKL_R"].fillna(merged_df["V00XRKL_L"])
merged_df["V00XRKL_L"] = merged_df["V00XRKL_L"].fillna(merged_df["V00XRKL_R"])

#similarly for the structure variables
for comp in ["JSL", "JSM"]:  # lateral, medial
    merged_df[f"V00XR{comp}_R"] = merged_df[f"V00XR{comp}_R"].fillna(merged_df[f"V00XR{comp}_L"])
    merged_df[f"V00XR{comp}_L"] = merged_df[f"V00XR{comp}_L"].fillna(merged_df[f"V00XR{comp}_R"])


In [23]:
#after all these operations we can drop the remaining na values given its only one row
merged_df.dropna(inplace=True)

In [24]:
#as you can see no missing values anymore
merged_df.isna().sum().sort_values(ascending=False).head(10)

ID             0
V00XROSTM_R    0
V00XRSCTL_R    0
V00XRSCTL_L    0
V00XROSFM_R    0
V00XROSFM_L    0
V00XROSFL_R    0
V00XROSFL_L    0
V00XROSTM_L    0
V00WOMTSR      0
dtype: int64

In [25]:
# i need to map the kl grades to numerical values ['2: 2', '1: 1', '3: 3', '0: 0', '4: 4'] to [2,1,3,0,4]
kl_mapping = {'0: 0': 0, '1: 1': 1, '2: 2': 2, '3: 3': 3, '4: 4': 4}
merged_df['V00XRKL_R'] = merged_df['V00XRKL_R'].map(kl_mapping)
merged_df['V00XRKL_L'] = merged_df['V00XRKL_L'].map(kl_mapping)


In [26]:
c = [
    "V00XRSCFM_L","V00XRSCFM_R","V00XRSCFL_L","V00XRSCFL_R",
    "V00XRSCTM_L","V00XRSCTM_R","V00XRSCTL_L","V00XRSCTL_R",
    "V00XROSFM_L","V00XROSFM_R","V00XROSFL_L","V00XROSFL_R",
    "V00XROSTM_L","V00XROSTM_R","V00XROSTL_L","V00XROSTL_R",
]
c_j = ["V00XRJSM_L","V00XRJSM_R","V00XRJSL_L","V00XRJSL_R"]

MISSING = ".: Missing Form/Incomplete Workbook"
CODE_MAP = {"0: 0": 0, "1: 1": 1, "2: 2": 2, "3: 3": 3}

def clean_xr_columns(df: pd.DataFrame, cols, code_map=CODE_MAP) -> None:
    cols = list(cols)
    present = [col for col in cols if col in df.columns]
    missing = [col for col in cols if col not in df.columns]

    if missing:
        print(f"Warning: {len(missing)} columns not found: {missing}")

    if not present:
        return

    # Replace the sentinel with NA (vectorized)
    df[present] = df[present].replace(MISSING, pd.NA)

    # Map the "k: k" strings to ints (vectorized). Anything else becomes NA.
    df[present] = df[present].replace(code_map)

    # Cast to pandas nullable integer
    df[present] = df[present].astype("Int64")

# Clean both sets
clean_xr_columns(merged_df, c)
clean_xr_columns(merged_df, c_j, code_map={}) 

In [27]:
merged_df.isna().sum().sort_values(ascending=False).head(20)

V00XRSCTM_R    1758
V00XRSCFM_R    1758
V00XRSCFL_R    1758
V00XRSCTL_R    1758
V00XRSCTL_L    1748
V00XRSCFM_L    1748
V00XRSCFL_L    1748
V00XRSCTM_L    1748
V00XROSTL_R    1741
V00XROSTM_R    1741
V00XROSFL_R    1741
V00XROSFM_R    1741
V00XROSTL_L    1732
V00XROSTM_L    1732
V00XROSFM_L    1732
V00XROSFL_L    1732
V00XRKL_L         0
V00XRKL_R         0
V99ERKVSAF        0
V99ELKVSAF        0
dtype: int64

In [28]:
merged_df

,ID,V00WOMTSR,V00KOOSKPR,V00KOOSYMR,V00KOOSQOL,V00WOMTSL,V00KOOSKPL,V00AGE,V00ABCIRC,V00XRJSL_R,...,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L,V99ERKVSAF,V99ELKVSAF,P02SEX
0,9000099,14.0,77.8,67.9,25.0,0.0,100.0,59,96.8,0,...,2,1,0,1,1,2,3,0,0,1
1,9000296,0.0,100.0,100.0,100.0,0.0,100.0,69,104.3,0,...,0,1,1,0,0,2,3,0,0,1
2,9000622,20.9,75.0,82.1,50.0,0.0,100.0,71,98.9,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,2
3,9000798,0.0,100.0,100.0,43.8,30.0,59.4,56,109.0,0,...,2,0,2,0,2,1,4,0,0,1
4,9001104,33.0,69.4,60.7,37.5,14.0,100.0,72,111.1,0,...,0,2,0,0,0,3,1,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4502,9999365,27.0,58.3,82.1,18.8,26.0,72.2,56,109.3,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,0,0,1
4503,9999510,0.0,100.0,100.0,68.8,11.8,83.3,50,97.5,0,...,0,1,1,0,0,1,3,0,0,1
4504,9999862,0.0,97.2,100.0,93.8,0.0,97.2,61,95.6,0,...,0,1,1,1,0,2,2,0,0,2
4505,9999865,0.0,100.0,100.0,100.0,0.0,100.0,61,95.5,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,1,0,0,2


In [29]:
#save the final dataset
merged_df.to_csv('../csv/clinical00_cleaned.csv', index=False)